<a href="https://colab.research.google.com/github/Laurasgrv-colab/PRe/blob/main/CaseStudy4_toy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study : Toy example

This notebook includes my implementation of the SOUL algorithm and its results with...

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#@title Load modules.

In [ ]:
#@title Load Data.

This is ...

In [ ]:
def soul(log_p_grad_x, th0, x0_M, y_l, y_f, T, M, B, delta_step, gamma_step):
  """
    Stochastic Optimisation via Unadjusted Langevin (SOUL).

    Parameters:
    - log_p_grad_x: Function returning grad_x log p(th, X, y). Shape matches X.
    - log_p_grad_th: Function returning grad_theta log p(th, X, y). Shape matches theta.
    - th_0: Initial parameters
    - x0_M: Initial latent variables from the previous step
    - y: Observed data
    - T: Number of outer optimization steps
    - M: Number of Langevin steps
    - B: Burn-in steps
    - delta_step: Step size for theta update
    - gamma_step: Langevin step size
    """
  th = np.copy(th0)
  x_t = np.copy(x0_M[:, 0:1]).reshape(D,1) # shape (D,1) (instead of (D,)) - keeps the column dimension

  x_values = np.array(x0_M)
  th_list = [th0]
  for t in range(1, T+1):

    # ULA steps
    for m in range(1, M+1):
      grad_x = log_p_grad_x(th, x_t, y_l, y_f)
      Z_k = np.random.normal(0.0, 1.0, x_t.shape) # Force size

      # Compute new x
      x_t += gamma_step * grad_x + np.sqrt(2.0 * gamma_step) * Z_k
      x_values = np.append(x_values, np.copy(x_t), axis=1) # Store new position

    burnin_x_samples = x_values[:, -(M-B):] # Shape (D, M-B)

    # Compute new theta
    avg_grad_th = np.zeros_like(th) # th is (1,1), so avg_grad_th is (1,1)

    # Iterate over particles (columns) instead of feature dimensions (rows)
    for idx in range(M-B):
      x_m_burnin = burnin_x_samples[:, idx:idx+1] # Shape (D,1)
      # (x_m_burnin - th) will broadcast to (D, 1).
      # We first sum only along D with .sum(0), then average over M-B.
      avg_grad_th += ((x_m_burnin - th).sum(0) / 5)

    th = th + delta_step * avg_grad_th /(M-B) # Update theta
    th_list.append(np.copy(th)) # Store new theta

  return th_list, x_values